# Chuẩn hóa địa chỉ quán ăn


In [5]:
%pip install pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Tách các địa điểm có nhiều địa chỉ thành các dòng khác nhau

In [6]:

import pandas as pd
import re
from pathlib import Path

# =====================================
# 1. KHAI BÁO FILE
# =====================================

INPUT_FILE = r"D:\VANH\School\ĐỒ ÁN\dataset\Food_tour_SG.xlsx"

# Ghi đè trực tiếp file đã tách dòng trước đó
OUTPUT_FILE = r"D:\VANH\School\ĐỒ ÁN\dataset\Food_tour_SG_tach_dong_dia_chi.xlsx"

SHEET_NAME = "Ban goc Food tour"
ADDRESS_COL = "Tên đường"


# =====================================
# 2. ĐỌC FILE EXCEL
# =====================================

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"Không tìm thấy file đầu vào: {INPUT_FILE}"
    )

sheets = pd.read_excel(
    INPUT_FILE,
    sheet_name=None
)

df = sheets[SHEET_NAME].copy()

if ADDRESS_COL not in df.columns:
    raise ValueError(
        f"Không tìm thấy cột {ADDRESS_COL}"
    )


# =====================================
# 3. HÀM NHẬN DIỆN DẤU GẠCH NGANG
# =====================================

def tach_dia_chi(text):

    # Ô trống thì giữ nguyên
    if pd.isna(text):
        return [text]

    text = str(text).strip()

    if not text:
        return [None]

    # Bước 1: Tách theo xuống dòng trước
    cac_doan = re.split(r"\r\n|\r|\n", text)

    danh_sach_dia_chi = []

    for doan in cac_doan:

        doan = doan.strip()

        if not doan:
            continue

        # Bước 2: Tách theo dấu gạch ngang
        # Chỉ tách khi dấu - có khoảng trắng hai bên
        # Ví dụ: địa chỉ A - địa chỉ B
        cac_dia_chi = re.split(
            r"\s+[-–—]\s+",
            doan
        )

        for dia_chi in cac_dia_chi:

            dia_chi = dia_chi.strip()

            if dia_chi:
                danh_sach_dia_chi.append(dia_chi)

    if not danh_sach_dia_chi:
        return [None]

    return danh_sach_dia_chi


# =====================================
# 4. TÁCH ĐỊA CHỈ THÀNH NHIỀU DÒNG
# =====================================

df["Danh sách địa chỉ"] = (
    df[ADDRESS_COL].apply(tach_dia_chi)
)

df_new = df.explode(
    "Danh sách địa chỉ",
    ignore_index=True
)

# Ghi địa chỉ đã tách vào cột ban đầu
df_new[ADDRESS_COL] = df_new["Danh sách địa chỉ"]

# Xóa cột phụ
df_new.drop(
    columns=["Danh sách địa chỉ"],
    inplace=True
)


# =====================================
# 5. CẬP NHẬT SHEET
# =====================================

sheets[SHEET_NAME] = df_new


# =====================================
# 6. GHI ĐÈ FILE ĐẦU RA
# =====================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    for sheet_name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


# =====================================
# 7. THỐNG KÊ KẾT QUẢ
# =====================================

print("ĐÃ TÁCH ĐỊA CHỈ THÀNH CÔNG!")
print("------------------------------------")
print(f"Số dòng ban đầu: {len(df)}")
print(f"Số dòng sau khi tách: {len(df_new)}")
print(f"Số dòng tăng thêm: {len(df_new) - len(df)}")
print("------------------------------------")
print(f"File đã được ghi đè: {OUTPUT_FILE}")


ĐÃ TÁCH ĐỊA CHỈ THÀNH CÔNG!
------------------------------------
Số dòng ban đầu: 440
Số dòng sau khi tách: 471
Số dòng tăng thêm: 31
------------------------------------
File đã được ghi đè: D:\VANH\School\ĐỒ ÁN\dataset\Food_tour_SG_tach_dong_dia_chi.xlsx


Chuẩn hóa Phường Quận về cùng format

In [7]:

INPUT_FILE = r"D:\VANH\School\ĐỒ ÁN\dataset\Food_tour_SG_tach_dong_dia_chi.xlsx"

OUTPUT_FILE = r"D:\VANH\School\ĐỒ ÁN\dataset\Food_tour_SG_chuan_hoa_phuong_quan.xlsx"

SHEET_NAME = "Ban goc Food tour"

ADDRESS_COL = "Tên đường"
DISTRICT_COL = "Quận"


# =====================================
# 2. KIỂM TRA VÀ ĐỌC FILE
# =====================================

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: {INPUT_FILE}"
    )

# Đọc tất cả sheet để giữ nguyên dữ liệu
sheets = pd.read_excel(
    INPUT_FILE,
    sheet_name=None
)

df = sheets[SHEET_NAME].copy()

# Kiểm tra các cột cần xử lý
for col in [ADDRESS_COL, DISTRICT_COL]:
    if col not in df.columns:
        raise ValueError(
            f"Không tìm thấy cột '{col}'. "
            f"Các cột hiện có: {list(df.columns)}"
        )


# =====================================
# 3. HÀM CHUẨN HÓA PHƯỜNG
# =====================================

def chuan_hoa_phuong(text):

    if pd.isna(text):
        return text

    s = str(text)

    # Chuẩn hóa khoảng trắng
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)

    # P.3 / P3 / P. 3 / P 3 -> Phường 3
    s = re.sub(
        r"(?i)(?<![\wÀ-ỹĐđ])P\s*\.?\s*(\d+)(?!\w)",
        r"Phường \1",
        s
    )

    # P. Bến Nghé -> Phường Bến Nghé
    s = re.sub(
        r"(?i)(?<![\wÀ-ỹĐđ])P\s*\.\s*"
        r"([A-Za-zÀ-ỹĐđ][A-Za-zÀ-ỹĐđ\s'-]*?)"
        r"(?=\s*[,;\n]|$)",
        lambda m: "Phường " + m.group(1).strip(),
        s
    )

    # P Bến Nghé -> Phường Bến Nghé
    s = re.sub(
        r"(?i)(?<![\wÀ-ỹĐđ])P\s+"
        r"([A-Za-zÀ-ỹĐđ][A-Za-zÀ-ỹĐđ\s'-]*?)"
        r"(?=\s*[,;\n]|$)",
        lambda m: "Phường " + m.group(1).strip(),
        s
    )

    # Chuẩn hóa cách viết Phường
    s = re.sub(
        r"(?i)(?<!\w)phường\s*",
        "Phường ",
        s
    )

    # Chuẩn hóa tên Đa Kao
    s = re.sub(
        r"(?i)\bĐaKao\b",
        "Đa Kao",
        s
    )

    s = re.sub(
        r"(?i)\bĐa\s+Kao\b",
        "Đa Kao",
        s
    )

    # Chuẩn hóa dấu phẩy và khoảng trắng
    s = re.sub(r"[ \t]*,[ \t]*", ", ", s)
    s = re.sub(r"[ \t]{2,}", " ", s)

    return s.strip()


# =====================================
# 4. HÀM CHUẨN HÓA QUẬN
# =====================================

def chuan_hoa_quan(text):

    if pd.isna(text):
        return text

    s = str(text).strip()

    # Chuẩn hóa khoảng trắng
    s = re.sub(r"\s+", " ", s)

    # Loại bỏ dấu chấm, dấu phẩy cuối
    s = s.strip(" .,;")

    # Giữ nguyên các trường hợp không phải quận
    if re.match(
        r"(?i)^(TP\.?|Thành phố)\s+",
        s
    ):
        return s

    # Bỏ tiền tố Q, Q., Quận nếu đã có
    s = re.sub(
        r"(?i)^(quận|q)\s*\.?\s*",
        "",
        s
    ).strip()

    # Chuẩn hóa chữ hoa/thường của phần tên
    if not s:
        return s

    # Thêm tiền tố Quận
    return "Quận " + s


# =====================================
# 5. ÁP DỤNG CHUẨN HÓA
# =====================================

# Chuẩn hóa phường trong cột Tên đường
df["Tên đường chuẩn hóa"] = (
    df[ADDRESS_COL].apply(chuan_hoa_phuong)
)

# Chuẩn hóa quận trong cột Quận
df["Quận chuẩn hóa"] = (
    df[DISTRICT_COL].apply(chuan_hoa_quan)
)


# =====================================
# 6. THỐNG KÊ KẾT QUẢ
# =====================================

so_dong = len(df)

so_phuong_thay_doi = (
    df[ADDRESS_COL].fillna("").astype(str)
    != df["Tên đường chuẩn hóa"].fillna("").astype(str)
).sum()

so_quan_thay_doi = (
    df[DISTRICT_COL].fillna("").astype(str)
    != df["Quận chuẩn hóa"].fillna("").astype(str)
).sum()


# =====================================
# 7. XUẤT FILE EXCEL
# =====================================

# Cập nhật sheet đã xử lý
sheets[SHEET_NAME] = df

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    for sheet_name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


# =====================================
# 8. IN KẾT QUẢ
# =====================================

print("ĐÃ CHUẨN HÓA PHƯỜNG VÀ QUẬN!")
print("------------------------------------")

print(f"Tổng số dòng: {so_dong}")

print(f"Số địa chỉ có thay đổi phường: {so_phuong_thay_doi}")

print(f"Số địa chỉ có thay đổi quận: {so_quan_thay_doi}")

print("------------------------------------")
print("File kết quả:")
print(OUTPUT_FILE)


ĐÃ CHUẨN HÓA PHƯỜNG VÀ QUẬN!
------------------------------------
Tổng số dòng: 471
Số địa chỉ có thay đổi phường: 204
Số địa chỉ có thay đổi quận: 7
------------------------------------
File kết quả:
D:\VANH\School\ĐỒ ÁN\dataset\Food_tour_SG_chuan_hoa_phuong_quan.xlsx
